# YOLO + ByteTrack HOTA

Full-sequence tracking evaluation for `player`, `goalkeeper`, and `referee`.

This notebook does **not** use team classification, PRTReID, or team labels.

In [1]:
from pathlib import Path
import configparser
import json
import shutil

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'outputs').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
RAW_TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'tracking' / 'train'
YOLO_WEIGHTS = PROJECT_ROOT / 'outputs' / 'detect_player' / 'runs' / 'E1_yolo_fullframe_img960' / 'weights' / 'best.pt'
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / 'tracking_hota'
RUNS_DIR = EXPERIMENT_DIR / 'runs'

print('Project root:', PROJECT_ROOT)
print('Raw train dir:', RAW_TRAIN_DIR)
print('YOLO weights:', YOLO_WEIGHTS)

Project root: C:\Users\CPU13374\Downloads\SoccerNet
Raw train dir: C:\Users\CPU13374\Downloads\SoccerNet\data\raw\tracking\train
YOLO weights: C:\Users\CPU13374\Downloads\SoccerNet\outputs\detect_player\runs\E1_yolo_fullframe_img960\weights\best.pt


In [2]:
SEQUENCE = 'SNMOT-060'
MAX_FRAMES = None  # Full sequence only. Keep None for the real experiment.
IMG_SIZE = 960
CONF = 0.05
IOU = 0.60
DEVICE = 0  # Use 'cpu' if CUDA is unavailable.

CLASS_NAMES = ['player', 'goalkeeper', 'referee']
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

seq_dir = RAW_TRAIN_DIR / SEQUENCE
img_dir = seq_dir / 'img1'
run_dir = RUNS_DIR / SEQUENCE
pred_dir = run_dir / 'preds'
trackeval_dir = run_dir / 'trackeval'
metrics_json = run_dir / 'metrics.json'
metrics_csv = run_dir / 'metrics.csv'

assert seq_dir.exists(), f'Missing sequence: {seq_dir}'
assert img_dir.exists(), f'Missing image folder: {img_dir}'
assert YOLO_WEIGHTS.exists(), f'Missing YOLO weights: {YOLO_WEIGHTS}'

run_dir.mkdir(parents=True, exist_ok=True)
pred_dir.mkdir(parents=True, exist_ok=True)
trackeval_dir.mkdir(parents=True, exist_ok=True)

print('Sequence:', SEQUENCE)
print('Run dir:', run_dir)

Sequence: SNMOT-060
Run dir: C:\Users\CPU13374\Downloads\SoccerNet\experiments\tracking_hota\runs\SNMOT-060


In [7]:
try:
    import trackeval
    TRACK_EVAL_AVAILABLE = True
    print('TrackEval is available')
except Exception as exc:
    TRACK_EVAL_AVAILABLE = False
    print('TrackEval is not installed.')
    print('Install it with: pip install git+https://github.com/JonathonLuiten/TrackEval.git')
    print('Import error:', repr(exc))

TrackEval is available


In [8]:
def read_seqinfo(seq_dir: Path):
    parser = configparser.ConfigParser()
    parser.read(seq_dir / 'seqinfo.ini')
    info = parser['Sequence']
    return {
        'name': info.get('name', seq_dir.name),
        'frame_rate': info.getint('frameRate'),
        'seq_length': info.getint('seqLength'),
        'width': info.getint('imWidth'),
        'height': info.getint('imHeight'),
        'im_ext': info.get('imExt', '.jpg'),
    }


def read_track_id_to_class(seq_dir: Path):
    parser = configparser.ConfigParser(strict=False)
    parser.optionxform = str
    parser.read(seq_dir / 'gameinfo.ini')
    mapping = {}
    if 'Sequence' not in parser:
        return mapping
    for key, value in parser['Sequence'].items():
        if not key.startswith('trackletID_'):
            continue
        track_id = int(key.replace('trackletID_', ''))
        desc = value.split(';')[0].strip().lower()
        if 'referee' in desc:
            cls = 'referee'
        elif 'goalkeeper' in desc or 'goalkeepers' in desc:
            cls = 'goalkeeper'
        elif 'player' in desc:
            cls = 'player'
        else:
            cls = None
        if cls in CLASS_TO_ID:
            mapping[track_id] = cls
    return mapping


def load_gt(seq_dir: Path):
    track_to_class = read_track_id_to_class(seq_dir)
    gt = pd.read_csv(seq_dir / 'gt' / 'gt.txt', header=None)
    gt.columns = ['frame', 'track_id', 'x', 'y', 'w', 'h', 'mark', 'c1', 'c2', 'c3'][:gt.shape[1]]
    gt['class_name'] = gt['track_id'].map(track_to_class)
    gt = gt[gt['class_name'].isin(CLASS_NAMES)].copy()
    gt['class_id'] = gt['class_name'].map(CLASS_TO_ID)
    return gt


seq_info = read_seqinfo(seq_dir)
gt_df = load_gt(seq_dir)
print(seq_info)
print(gt_df['class_name'].value_counts())
print('GT rows after filtering:', len(gt_df))

{'name': 'SNMOT-060', 'frame_rate': 25, 'seq_length': 750, 'width': 1920, 'height': 1080, 'im_ext': '.jpg'}
class_name
player        11079
referee        1439
goalkeeper      288
Name: count, dtype: int64
GT rows after filtering: 12806


In [9]:
import tqdm


def run_yolo_bytetrack(img_dir: Path, output_txt: Path):
    image_paths = sorted(img_dir.glob(f'*{seq_info["im_ext"]}'))
    if MAX_FRAMES is not None:
        image_paths = image_paths[:MAX_FRAMES]
    assert image_paths, f'No images found in {img_dir}'

    model = YOLO(str(YOLO_WEIGHTS))
    rows = []
    from tqdm import tqdm
    for frame_idx, image_path in tqdm(enumerate(image_paths, start=1), total=len(image_paths)):
        result = model.track(
            str(image_path),
            tracker='bytetrack.yaml',
            persist=True,
            imgsz=IMG_SIZE,
            conf=CONF,
            iou=IOU,
            device=DEVICE,
            verbose=False,
        )[0]

        boxes = result.boxes
        if boxes is None or boxes.id is None:
            continue

        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.cpu().numpy().astype(int)
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()

        for box, track_id, cls_id, score in zip(xyxy, ids, cls_ids, confs):
            if int(cls_id) not in CLASS_TO_ID.values():
                continue
            x1, y1, x2, y2 = map(float, box)
            rows.append({
                'frame': frame_idx,
                'track_id': int(track_id),
                'x': x1,
                'y': y1,
                'w': max(0.0, x2 - x1),
                'h': max(0.0, y2 - y1),
                'score': float(score),
                'class_id': int(cls_id),
                'class_name': CLASS_NAMES[int(cls_id)],
            })

    pred = pd.DataFrame(rows)
    if pred.empty:
        raise RuntimeError('ByteTrack produced no tracked boxes with track_id.')

    output_txt.parent.mkdir(parents=True, exist_ok=True)
    pred[['frame', 'track_id', 'x', 'y', 'w', 'h', 'score', 'class_id']].to_csv(
        output_txt, header=False, index=False, float_format='%.3f'
    )
    return pred


pred_txt = pred_dir / 'bytetrack.txt'
pred_df = run_yolo_bytetrack(img_dir, pred_txt)
print('Prediction txt:', pred_txt)
print(pred_df['class_name'].value_counts())
print('Pred rows:', len(pred_df))

  0%|          | 0/750 [00:00<?, ?it/s]

100%|██████████| 750/750 [00:12<00:00, 60.78it/s]

Prediction txt: C:\Users\CPU13374\Downloads\SoccerNet\experiments\tracking_hota\runs\SNMOT-060\preds\bytetrack.txt
class_name
player        10845
referee        1307
goalkeeper      278
Name: count, dtype: int64
Pred rows: 12430


In [12]:
def write_seqinfo(target_seq_dir: Path):
    text = (seq_dir / 'seqinfo.ini').read_text(encoding='utf-8')
    target_seq_dir.mkdir(parents=True, exist_ok=True)
    (target_seq_dir / 'seqinfo.ini').write_text(text, encoding='utf-8')


def write_trackeval_subset(class_name: str):
    class_root = trackeval_dir / class_name
    if class_root.exists():
        shutil.rmtree(class_root)

    gt_seq_dir = class_root / 'gt' / SEQUENCE
    tracker_data_dir = class_root / 'trackers' / 'bytetrack' / 'data'
    (gt_seq_dir / 'gt').mkdir(parents=True, exist_ok=True)
    tracker_data_dir.mkdir(parents=True, exist_ok=True)
    write_seqinfo(gt_seq_dir)

    gt_part = gt_df[gt_df['class_name'] == class_name].copy()
    gt_out = pd.DataFrame({
        0: gt_part['frame'].astype(int),
        1: gt_part['track_id'].astype(int),
        2: gt_part['x'],
        3: gt_part['y'],
        4: gt_part['w'],
        5: gt_part['h'],
        6: 1,
        7: 1,  # TrackEval MOTChallenge class id: pedestrian
        8: 1,
    })
    gt_out.to_csv(gt_seq_dir / 'gt' / 'gt.txt', header=False, index=False, float_format='%.3f')

    pred_part = pred_df[pred_df['class_name'] == class_name].copy()
    pred_out = pd.DataFrame({
        0: pred_part['frame'].astype(int),
        1: pred_part['track_id'].astype(int),
        2: pred_part['x'],
        3: pred_part['y'],
        4: pred_part['w'],
        5: pred_part['h'],
        6: pred_part['score'],
        7: -1,
        8: -1,
        9: -1,
    })
    pred_out.to_csv(tracker_data_dir / f'{SEQUENCE}.txt', header=False, index=False, float_format='%.3f')
    return class_root


subset_roots = {class_name: write_trackeval_subset(class_name) for class_name in CLASS_NAMES}
subset_roots

{'player': WindowsPath('C:/Users/CPU13374/Downloads/SoccerNet/experiments/tracking_hota/runs/SNMOT-060/trackeval/player'),
 'goalkeeper': WindowsPath('C:/Users/CPU13374/Downloads/SoccerNet/experiments/tracking_hota/runs/SNMOT-060/trackeval/goalkeeper'),
 'referee': WindowsPath('C:/Users/CPU13374/Downloads/SoccerNet/experiments/tracking_hota/runs/SNMOT-060/trackeval/referee')}

In [13]:
if not TRACK_EVAL_AVAILABLE:
    raise ImportError('TrackEval is required. Run: pip install git+https://github.com/JonathonLuiten/TrackEval.git')

# TrackEval still references removed NumPy aliases in some installs.
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'int'):
    np.int = int


def mean_value(value):
    arr = np.asarray(value, dtype=float)
    return float(np.nanmean(arr))


def evaluate_one_class(class_name: str, class_root: Path):
    eval_config = trackeval.Evaluator.get_default_eval_config()
    eval_config.update({
        'PRINT_RESULTS': False,
        'PRINT_ONLY_COMBINED': True,
        'DISPLAY_LESS_PROGRESS': True,
        'OUTPUT_SUMMARY': False,
        'OUTPUT_DETAILED': False,
        'PLOT_CURVES': False,
    })

    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    dataset_config.update({
        'GT_FOLDER': str(class_root / 'gt'),
        'TRACKERS_FOLDER': str(class_root / 'trackers'),
        'OUTPUT_FOLDER': str(class_root / 'trackeval_output'),
        'TRACKERS_TO_EVAL': ['bytetrack'],
        'CLASSES_TO_EVAL': ['pedestrian'],
        'BENCHMARK': 'MOT17',
        'SPLIT_TO_EVAL': 'train',
        'INPUT_AS_ZIP': False,
        'PRINT_CONFIG': False,
        'DO_PREPROC': False,
        'TRACKER_SUB_FOLDER': 'data',
        'OUTPUT_SUB_FOLDER': '',
        'SEQ_INFO': {SEQUENCE: seq_info['seq_length']},
        'SKIP_SPLIT_FOL': True,
    })

    metrics_config = {'METRICS': ['HOTA', 'CLEAR', 'Identity'], 'THRESHOLD': 0.5}
    evaluator = trackeval.Evaluator(eval_config)
    dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)
    metrics = [
        trackeval.metrics.HOTA(metrics_config),
        trackeval.metrics.CLEAR(metrics_config),
        trackeval.metrics.Identity(metrics_config),
    ]

    results, _ = evaluator.evaluate([dataset], metrics)
    combined = results['MotChallenge2DBox']['bytetrack']['COMBINED_SEQ']['pedestrian']
    row = {'class': class_name}
    for metric_name, metric_values in combined.items():
        if isinstance(metric_values, dict):
            for key, value in metric_values.items():
                row[key] = mean_value(value)
    return row


metric_rows = [evaluate_one_class(class_name, subset_roots[class_name]) for class_name in CLASS_NAMES]
metrics_df = pd.DataFrame(metric_rows)
mean_row = metrics_df.select_dtypes(include='number').mean().to_dict()
mean_row['class'] = 'mean'
metrics_df = pd.concat([metrics_df, pd.DataFrame([mean_row])], ignore_index=True)

metrics_df.to_csv(metrics_csv, index=False)
metrics_json.write_text(json.dumps(metric_rows + [mean_row], indent=2), encoding='utf-8')

print('Saved:', metrics_csv)
print('Saved:', metrics_json)
metrics_df


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : c:\Users\CPU13374\AppData\Local\Programs\Python\Python310\lib\site-packages\error_log.txt
PRINT_RESULTS        : False                         
PRINT_ONLY_COMBINED  : True                          
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : True                          
OUTPUT_SUMMARY       : False                         
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : False                         
PLOT_CURVES          : False                         

CLEAR Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                    

,class,HOTA_TP,HOTA_FN,HOTA_FP,AssRe,AssPr,AssA,LocA,DetRe,DetPr,...,IDTP,IDFN,IDFP,IDR,IDP,IDF1,Dets,GT_Dets,IDs,GT_IDs
0,player,8941.947368,2137.052632,1903.052632,0.595975,0.812234,0.566381,0.840510,0.807108,0.824523,...,8790.000000,2289.000000,2055.0,0.793393,0.810512,0.801861,10845.000000,11079.000000,68.0,20.000000
1,goalkeeper,227.736842,60.263158,50.263158,0.793846,0.820695,0.756226,0.830990,0.790753,0.819197,...,277.000000,11.000000,1.0,0.961806,0.996403,0.978799,278.000000,288.000000,2.0,2.000000
2,referee,1046.631579,392.368421,260.368421,0.504295,0.814098,0.483842,0.825978,0.727333,0.800789,...,993.000000,446.000000,314.0,0.690063,0.759755,0.723234,1307.000000,1439.000000,11.0,3.000000
3,mean,3405.438596,863.228070,737.894737,0.631372,0.815675,0.602150,0.832493,0.775064,0.814836,...,3353.333333,915.333333,790.0,0.815087,0.855557,0.834631,4143.333333,4268.666667,27.0,8.333333
